# Token Confusion Analysis

Systematic prediction error patterns: confusion matrices, logit competition, layer-resolved confusion, systematic errors, and position-dependent error modes.

## Why This Matters

Token confusion tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_confusion import (
    prediction_confusion_matrix, logit_competition,
    layer_resolved_confusion, systematic_errors,
    position_error_modes,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
tokens_list = [jnp.array([i, i+10, i+20, i+30, i+40, i+50, i+60]) for i in range(1, 20)]
print('Model ready')

In [ ]:
result = prediction_confusion_matrix(model, tokens_list, pos=-2)
print(f'Accuracy: {result["accuracy"]:.2%}')
print(f'Top confused pairs:')
for c in result['top_confused'][:5]:
    print(f'  actual={c["actual"]}, predicted={c["predicted"]}, count={c["count"]}')

In [ ]:
result = logit_competition(model, tokens)
print(f'Competition score: {result["competition_score"]:.3f}')
print(f'Entropy: {result["entropy"]:.3f}')
for t in result['top_tokens']:
    print(f'  Token {t["token"]}: logit={t["logit"]:.3f}, prob={t["probability"]:.4f}')

In [ ]:
result = layer_resolved_confusion(model, tokens)
print(f'Transitions: {result["n_transitions"]}')
for p in result['per_layer']:
    print(f'  Stage {p["stage"]}: top={p["top_token"]}, gap={p["gap"]:.3f}')

In [ ]:
result = systematic_errors(model, tokens_list, pos=-2, min_occurrences=1)
print(f'Mean error rate: {result["mean_error_rate"]:.2%}')
for t in result['per_token'][:5]:
    print(f'  Token {t["token"]}: error_rate={t["error_rate"]:.2%}')

In [ ]:
result = position_error_modes(model, tokens_list)
for p in result['per_position']:
    print(f'  Pos {p["position"]}: accuracy={p["accuracy"]:.2%}, entropy={p["avg_entropy"]:.3f}')